In [ ]:
# Cell 1 [全局模型规模开关 + 挂载 Google Drive 与 GPU 体检]
# ==============================================================================
# 单变量全局开关（供 Cell 3 / Cell 5 / Cell 6 自动读取）：
#   "210m" -> DCPA v2 全科认知 35:35:30（208.65M 总参 / 22.86M 激活，32 experts Top-2，零 Rust）
#   "48m"  -> DCPA v1 基线（47.62M 总参 / 15.77M 激活，8 experts Top-2）
# CORPUS_SCALE_FACTOR (210m 生效): 12.0 = 单卡正式 ~5.28 GB 底座语料; 2.0 = ~880 MB 轻量版
# ==============================================================================
MODEL_PRESET = "210m"        # 默认推荐 DCPA v2 210M；可切回 "48m"
CORPUS_SCALE_FACTOR = 12.0   # 若 Drive 空间不足可降至 4.0 或 2.0
SMOKE_TEST_ONLY = False      # True = 2 分钟全链路冒烟验证

import os
import subprocess
from pathlib import Path

if MODEL_PRESET not in ("48m", "210m"):
    raise ValueError(f"MODEL_PRESET must be '48m' or '210m', got {MODEL_PRESET!r}")
os.environ["DCPA_MODEL_PRESET"] = MODEL_PRESET
os.environ["DCPA_CORPUS_SCALE"] = str(CORPUS_SCALE_FACTOR)
os.environ["DCPA_SMOKE_ONLY"] = "1" if SMOKE_TEST_ONLY else "0"

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive/DCPA_Project")
for sub in ("code", "data_archive", "checkpoints", "logs", "export"):
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)
print(
    f"[Drive] Ready: {DRIVE_ROOT} | MODEL_PRESET={MODEL_PRESET} | "
    f"CORPUS_SCALE_FACTOR={CORPUS_SCALE_FACTOR} | SMOKE_TEST_ONLY={SMOKE_TEST_ONLY}"
)

# GPU health check & 4-tier hardware sharding preview (no hardcoded torch.cuda.get_device_properties)
smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True,
    text=True,
    check=False,
)
if smi.returncode == 0 and smi.stdout.strip():
    first_line = smi.stdout.strip().splitlines()[0]
    gpu_name, mem_mib_s = [part.strip() for part in first_line.split(",", 1)]
    vram_gb = float(mem_mib_s) / 1024.0
    if vram_gb >= 88.0:
        tier, micro_bytes, accum = "blackwell_or_h200_96g", 262144, 1
    elif vram_gb >= 75.0:
        tier, micro_bytes, accum = "h100_or_a100_80g", 131072, 1
    elif vram_gb >= 38.0:
        tier, micro_bytes, accum = "a100_40g", 65536, 2
    elif vram_gb >= 20.0:
        tier, micro_bytes, accum = "l4_or_4090", 32768, 4
    else:
        tier, micro_bytes, accum = "cuda_compact", 16384, 8
    print(
        f"[GPU Health] {gpu_name} | VRAM={vram_gb:.1f} GB | tier={tier} | "
        f"micro_batch_bytes={micro_bytes} | grad_accum_steps={accum} | precision=bfloat16"
    )
else:
    print("[GPU Health][WARN] No NVIDIA GPU detected via nvidia-smi; will fall back to CPU float32.")


In [ ]:
# Cell 2 [部署代码至本地 NVMe 与安装依赖]
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

BUNDLE_ZIP = Path("/content/drive/MyDrive/DCPA_Project/code/dcpa_colab_bundle.zip")
CODE_DIR = Path("/content/dcpa")
NVME_CACHE = Path("/content/nvme_cache")

if not BUNDLE_ZIP.exists():
    raise FileNotFoundError(
        f"Missing code bundle at {BUNDLE_ZIP}. Run `bash scripts/pack_for_colab.sh` locally "
        "and upload `dcpa_colab_bundle.zip` to Google Drive first."
    )

if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)
CODE_DIR.mkdir(parents=True, exist_ok=True)
NVME_CACHE.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(BUNDLE_ZIP, "r") as zf:
    zf.extractall(CODE_DIR)

os.chdir(CODE_DIR)
for p in (str(CODE_DIR), str(CODE_DIR / "dcpa-train-py")):
    if p not in sys.path:
        sys.path.insert(0, p)
os.environ["PYTHONPATH"] = f"{CODE_DIR}:{CODE_DIR / 'dcpa-train-py'}"

deps = [
    "safetensors",
    "datasets",
    "hf_transfer",
    "tree-sitter",
    "tree-sitter-python",
    "tree-sitter-javascript",
    "tree-sitter-rust",
    "pyyaml",
    "pytest",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *deps], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--no-build-isolation", "-e", str(CODE_DIR / "dcpa-train-py")],
    check=True,
)

from utils.device import describe_hardware_profile, detect_hardware_profile

profile = detect_hardware_profile("auto")
print(f"[Deploy] Code unpacked to {CODE_DIR} | HF_HUB_ENABLE_HF_TRANSFER={os.environ['HF_HUB_ENABLE_HF_TRANSFER']}")
print(describe_hardware_profile(profile))


In [ ]:
# Cell 3 [全科认知 / 基线底座与能力语料智能缓存：首次流式构建归档，重连极速恢复]
import os
import shutil
import subprocess
import sys
import tarfile
import time
from pathlib import Path

from train.colab_drive_sync import sync_file_to_drive

MODEL_PRESET = os.environ.get("DCPA_MODEL_PRESET", "210m")
CORPUS_SCALE = os.environ.get("DCPA_CORPUS_SCALE", "12.0")
SMOKE_ONLY = os.environ.get("DCPA_SMOKE_ONLY", "0") == "1"

if MODEL_PRESET == "210m":
    suf = "_smoke" if SMOKE_ONLY else ""
    DRIVE_ARCHIVE = Path(f"/content/drive/MyDrive/DCPA_Project/data_archive/dcpa_corpus_v2_cognitive{suf}.tar.gz")
    DRIVE_CAP_ARCHIVE = Path(f"/content/drive/MyDrive/DCPA_Project/data_archive/dcpa_cap_v2_cognitive{suf}.tar.gz")
else:
    DRIVE_ARCHIVE = Path("/content/drive/MyDrive/DCPA_Project/data_archive/dcpa_corpus_v1.tar.gz")
    DRIVE_CAP_ARCHIVE = Path("/content/drive/MyDrive/DCPA_Project/data_archive/dcpa_cap_v1.tar.gz")

NVME_DATA_DIR = Path("/content/nvme_cache/data")
NVME_CAP_DIR = Path("/content/nvme_cache/cap_data")
NVME_TAR_TMP = Path(f"/content/nvme_cache/{DRIVE_ARCHIVE.name}")
NVME_CAP_TAR_TMP = Path(f"/content/nvme_cache/{DRIVE_CAP_ARCHIVE.name}")

NVME_DATA_DIR.mkdir(parents=True, exist_ok=True)
NVME_CAP_DIR.mkdir(parents=True, exist_ok=True)

def _stream_run(cmd, env=None, check=True):
    proc = subprocess.Popen(
        cmd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    rc = int(proc.wait())
    if check and rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)
    return rc

t0 = time.perf_counter()
smoke_flag = ["--smoke-quota"] if SMOKE_ONLY else []

# 1. Base pretraining corpus
if DRIVE_ARCHIVE.exists():
    print(f"[Corpus Cache HIT] Restoring {DRIVE_ARCHIVE} -> {NVME_DATA_DIR} ...")
    shutil.copy2(DRIVE_ARCHIVE, NVME_TAR_TMP)
    with tarfile.open(NVME_TAR_TMP, "r:gz") as tf:
        tf.extractall(NVME_DATA_DIR)
    NVME_TAR_TMP.unlink(missing_ok=True)
    print(f"[Corpus Cache HIT] Restored in {time.perf_counter() - t0:.2f}s.")
else:
    if MODEL_PRESET == "210m":
        print(f"[Corpus Cache MISS] Building 6-domain v2 holistic corpus (scale={CORPUS_SCALE}x, zero Rust) on NVMe...")
        _stream_run([
            sys.executable, "-u", "data/build_corpus.py",
            "--out-dir", str(NVME_DATA_DIR),
            "--cognitive-v2", "--workers", "4", "--scale-factor", CORPUS_SCALE,
            *smoke_flag,
        ])
    else:
        print("[Corpus Cache MISS] Building ~450 MB v1 base corpus directly on local NVMe...")
        _stream_run([sys.executable, "-u", "data/build_corpus.py", "--out-dir", str(NVME_DATA_DIR), *smoke_flag])
    env = dict(os.environ)
    env["DCPA_DATA_DIR"] = str(NVME_DATA_DIR)
    _stream_run([sys.executable, "-u", "-m", "pytest", "-v", "tests/test_data_pipeline.py"], env=env)
    with tarfile.open(NVME_TAR_TMP, "w:gz") as tf:
        for fname in ("train.bin", "train.idx", "train.meta.json", "val.bin", "val.idx", "val.meta.json", "corpus_meta.json"):
            fpath = NVME_DATA_DIR / fname
            if fpath.exists():
                tf.add(fpath, arcname=fname)
    sync_file_to_drive(NVME_TAR_TMP, DRIVE_ARCHIVE.parent, dest_name=DRIVE_ARCHIVE.name)
    NVME_TAR_TMP.unlink(missing_ok=True)
    print(f"[Corpus] Archived to {DRIVE_ARCHIVE} in {time.perf_counter() - t0:.1f}s.")

# 2. Capability & Data-Alchemy Shards
t1 = time.perf_counter()
if DRIVE_CAP_ARCHIVE.exists():
    print(f"[Cap Cache HIT] Restoring {DRIVE_CAP_ARCHIVE} -> {NVME_CAP_DIR} ...")
    shutil.copy2(DRIVE_CAP_ARCHIVE, NVME_CAP_TAR_TMP)
    with tarfile.open(NVME_CAP_TAR_TMP, "r:gz") as tf:
        tf.extractall(NVME_CAP_DIR)
    NVME_CAP_TAR_TMP.unlink(missing_ok=True)
    print(f"[Cap Cache HIT] Restored capability shards in {time.perf_counter() - t1:.2f}s.")
else:
    if MODEL_PRESET == "210m":
        print("[Cap Cache MISS] Building 520 MB v2 Alchemy shards (reasoning 160 + writing 160 + code 120 + instruct 40 + toolcall 40)...")
        _stream_run([sys.executable, "-u", "data/build_cap_reasoning.py", "--out-dir", str(NVME_CAP_DIR), "--target-mb", "160", "--cognitive-v2", *smoke_flag])
        _stream_run([sys.executable, "-u", "data/build_cap_writing.py", "--out-dir", str(NVME_CAP_DIR), "--target-mb", "160", *smoke_flag])
        _stream_run([sys.executable, "-u", "data/build_cap_code_v2.py", "--out-dir", str(NVME_CAP_DIR), "--target-mb", "120", *smoke_flag])
        _stream_run([sys.executable, "-u", "data/build_cap_instruct.py", "--out-dir", str(NVME_CAP_DIR), "--target-mb", "40", *smoke_flag])
        _stream_run([sys.executable, "-u", "data/build_cap_toolcall.py", "--out-dir", str(NVME_CAP_DIR), "--target-mb", "40", *smoke_flag])
    else:
        print("[Cap Cache MISS] Streaming 260 MB v1 capability shards (reasoning 100MB + writing 100MB + toolcall 60MB)...")
        _stream_run([sys.executable, "-u", "data/build_cap_reasoning.py", "--out-dir", str(NVME_CAP_DIR), "--target-mb", "100", *smoke_flag])
        _stream_run([sys.executable, "-u", "data/build_cap_writing.py", "--out-dir", str(NVME_CAP_DIR), "--target-mb", "100", *smoke_flag])
        _stream_run([sys.executable, "-u", "data/build_cap_toolcall.py", "--out-dir", str(NVME_CAP_DIR), "--target-mb", "60", *smoke_flag])
    with tarfile.open(NVME_CAP_TAR_TMP, "w:gz") as tf:
        for fpath in sorted(NVME_CAP_DIR.iterdir()):
            if fpath.is_file():
                tf.add(fpath, arcname=fpath.name)
    sync_file_to_drive(NVME_CAP_TAR_TMP, DRIVE_CAP_ARCHIVE.parent, dest_name=DRIVE_CAP_ARCHIVE.name)
    NVME_CAP_TAR_TMP.unlink(missing_ok=True)
    if hasattr(os, "sync"):
        os.sync()
    print(f"[Cap] Archived capability shards to {DRIVE_CAP_ARCHIVE} in {time.perf_counter() - t1:.1f}s.")


In [ ]:
# Cell 4 [云端 CUDA 算子与 DCPA v2 全科认知管线上线复检]
import subprocess
import sys
import time

t0 = time.perf_counter()
cmd = [
    sys.executable,
    "-u",
    "-m",
    "pytest",
    "-v",
    "tests/test_sanity.py",
    "tests/test_trainer_lifecycle.py",
    "tests/test_capabilities_upgrade.py",
    "tests/test_v2_cognitive_pipeline.py",
]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert proc.stdout is not None
for line in proc.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()
rc = int(proc.wait())
if rc != 0:
    raise RuntimeError(f"Cloud preflight pytest failed with code {rc}")
print(f"[Cloud Preflight] Causality, bf16 Islands, STE gradients, Capability Upgrade & v2 Cognitive 100% GREEN in {time.perf_counter() - t0:.1f}s.")


In [ ]:
# Cell 5 [一键启动 / 断连无缝续跑 10 万步底座训练（支持 v2 全科 210M / 基线 48M）]
# 注：自动释放显卡逻辑已移至 Cell 6（全部 Stage 1~5 任务完成后统一释放）
import os
import shutil
import subprocess
import sys
import tarfile
from pathlib import Path

MODEL_PRESET = os.environ.get("DCPA_MODEL_PRESET", "210m")
SMOKE_ONLY = os.environ.get("DCPA_SMOKE_ONLY", "0") == "1"
if MODEL_PRESET == "210m":
    suf = "_smoke" if SMOKE_ONLY else ""
    CFG_FILE, PRESET_NAME = "configs/dcpa_210m_v2_cognitive.yaml", "moe_210m"
    DRIVE_ARCHIVE_NAME = f"dcpa_corpus_v2_cognitive{suf}.tar.gz"
    DRIVE_CAP_NAME = f"dcpa_cap_v2_cognitive{suf}.tar.gz"
    CAP_SENTINEL = "cap_code_train.bin"
else:
    CFG_FILE, PRESET_NAME = "configs/dcpa_48m_train.yaml", "proto_48m"
    DRIVE_ARCHIVE_NAME = "dcpa_corpus_v1.tar.gz"
    DRIVE_CAP_NAME = "dcpa_cap_v1.tar.gz"
    CAP_SENTINEL = "cap_reasoning_train.bin"

DRIVE_ROOT = Path("/content/drive/MyDrive/DCPA_Project")
NVME_CACHE = Path("/content/nvme_cache")
NVME_DATA_DIR = NVME_CACHE / "data"
NVME_CAP_DIR = NVME_CACHE / "cap_data"
DRIVE_ARCHIVE = DRIVE_ROOT / "data_archive" / DRIVE_ARCHIVE_NAME
DRIVE_CAP_ARCHIVE = DRIVE_ROOT / "data_archive" / DRIVE_CAP_NAME
DRIVE_CKPTS = DRIVE_ROOT / "checkpoints"
DRIVE_LOGS = DRIVE_ROOT / "logs"
DRIVE_EXPORT = DRIVE_ROOT / "export"
FINAL_SAFETENSORS = DRIVE_EXPORT / "dcpa_final.safetensors"

# 断连重连防呆保护：若新实例 NVMe 尚未解压语料而 Drive 已有归档，自动恢复至本地 NVMe！
NVME_DATA_DIR.mkdir(parents=True, exist_ok=True)
NVME_CAP_DIR.mkdir(parents=True, exist_ok=True)
if not (NVME_DATA_DIR / "train.bin").exists() and DRIVE_ARCHIVE.exists():
    print(f"[Auto-Restore] NVMe train.bin missing; restoring {DRIVE_ARCHIVE} -> {NVME_DATA_DIR} ...", flush=True)
    tmp_tar = NVME_CACHE / "_auto_corpus.tar.gz"
    shutil.copy2(DRIVE_ARCHIVE, tmp_tar)
    with tarfile.open(tmp_tar, "r:gz") as tf:
        tf.extractall(NVME_DATA_DIR)
    tmp_tar.unlink(missing_ok=True)
if not (NVME_CAP_DIR / CAP_SENTINEL).exists() and DRIVE_CAP_ARCHIVE.exists():
    print(f"[Auto-Restore] NVMe cap_data missing; restoring {DRIVE_CAP_ARCHIVE} -> {NVME_CAP_DIR} ...", flush=True)
    tmp_cap = NVME_CACHE / "_auto_cap.tar.gz"
    shutil.copy2(DRIVE_CAP_ARCHIVE, tmp_cap)
    with tarfile.open(tmp_cap, "r:gz") as tf:
        tf.extractall(NVME_CAP_DIR)
    tmp_cap.unlink(missing_ok=True)

total_steps = 200 if SMOKE_ONLY else 100000
stage_args = ["--stage-boundaries", "0,30,90,150,200"] if SMOKE_ONLY else []
seq_args = [] if MODEL_PRESET == "210m" else ["--seq-len", "2048"]
save_every = 50 if SMOKE_ONLY else 1000
probe_every = 50 if SMOKE_ONLY else 500

cmd = [
    sys.executable,
    "-u",
    "train/train_dcpa.py",
    "--gpus", "1",
    "--config", CFG_FILE,
    "--preset", PRESET_NAME,
    "--total-steps", str(total_steps),
    *stage_args,
    *seq_args,
    "--data-dir", str(NVME_DATA_DIR),
    "--cap-data-dir", str(NVME_CAP_DIR),
    "--local-cache-dir", str(NVME_CACHE),
    "--drive-sync-dir", str(DRIVE_CKPTS),
    "--drive-logs-dir", str(DRIVE_LOGS),
    "--drive-export-dir", str(DRIVE_EXPORT),
    "--save-every-steps", str(save_every),
    "--save-every-minutes", "10",
    "--probe-every-steps", str(probe_every),
    "--heartbeat-seconds", "60",
    "--resume", "auto",
    "--compile-mode", "auto",
    "--max-consecutive-meltdowns", "3",
]

print(f"[Train][{MODEL_PRESET}] Launching: {' '.join(cmd)}", flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert proc.stdout is not None
for line in proc.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()
exit_code = int(proc.wait())
if hasattr(os, "sync"):
    os.sync()
if exit_code == 0:
    print(f"[Train] {total_steps:,} steps ({PRESET_NAME}) completed cleanly! Final export: {FINAL_SAFETENSORS}", flush=True)
elif exit_code == 42:
    raise RuntimeError("[Train][CIRCUIT_BREAKER] Aborted after 3 consecutive unrecoverable meltdowns.")
else:
    raise RuntimeError(f"Training exited unexpectedly with code {exit_code}")


In [ ]:
# Cell 6 [Stage 5：15,000 步全科多稿打磨 / 苏格拉底思辨 / 排错微调与导出 + 完工自动释放显卡]
# ==============================================================================
# AUTO_UNASSIGN_RUNTIME_ON_FINISH:
#   True  -> Stage 5 跑满后导出 dcpa_chat_sft.safetensors 至 Google Drive，
#            强制 os.sync() 等待 10 秒并自动归还 H100/A100 实例，防止空烧 CCU。
#   False -> 保留当前 Colab 实例以便手动调试。
# ==============================================================================
AUTO_UNASSIGN_RUNTIME_ON_FINISH = True

import os
import shutil
import subprocess
import sys
import tarfile
import time
from pathlib import Path

from train.colab_drive_sync import sync_file_to_drive

MODEL_PRESET = os.environ.get("DCPA_MODEL_PRESET", "210m")
SMOKE_ONLY = os.environ.get("DCPA_SMOKE_ONLY", "0") == "1"
DRIVE_ROOT = Path("/content/drive/MyDrive/DCPA_Project")
NVME_CACHE = Path("/content/nvme_cache")
NVME_SFT_DIR = NVME_CACHE / "sft_data"
NVME_CAP_DIR = NVME_CACHE / "cap_data"
if MODEL_PRESET == "210m":
    suf = "_smoke" if SMOKE_ONLY else ""
    DRIVE_SFT_ARCHIVE = DRIVE_ROOT / "data_archive" / f"dcpa_sft_stage5_v2_cognitive{suf}.tar.gz"
    SFT_STEPS = "50" if SMOKE_ONLY else "15000"
    MAX_DOC_BYTES = "4096"
else:
    DRIVE_SFT_ARCHIVE = DRIVE_ROOT / "data_archive" / "dcpa_sft_stage5_v1.tar.gz"
    SFT_STEPS = "50" if SMOKE_ONLY else "8000"
    MAX_DOC_BYTES = "2048"
DRIVE_EXPORT = DRIVE_ROOT / "export"
BASE_WEIGHTS = DRIVE_EXPORT / "dcpa_final.safetensors"
BASE_CONFIG = DRIVE_EXPORT / "config.json"
CHAT_WEIGHTS = DRIVE_EXPORT / "dcpa_chat_sft.safetensors"

exit_code = -1
try:
    if not BASE_WEIGHTS.exists():
        raise FileNotFoundError(f"Missing Stage 4 base export: {BASE_WEIGHTS}. Run Cell 5 first.")

    # 1. Build or restore Stage 5 SFT dataset on local NVMe
    NVME_SFT_DIR.mkdir(parents=True, exist_ok=True)
    if DRIVE_SFT_ARCHIVE.exists():
        tmp_tar = NVME_CACHE / "sft_stage5.tar.gz"
        shutil.copy2(DRIVE_SFT_ARCHIVE, tmp_tar)
        with tarfile.open(tmp_tar, "r:gz") as tf:
            tf.extractall(NVME_SFT_DIR)
        tmp_tar.unlink(missing_ok=True)
        print(f"[Stage 5 Data] Restored cached SFT corpus -> {NVME_SFT_DIR}", flush=True)
    else:
        build_sft_cmd = [sys.executable, "-u", "data/build_sft_stage5.py", "--out-dir", str(NVME_SFT_DIR)]
        if MODEL_PRESET == "210m":
            build_sft_cmd.extend(["--cognitive-v2", "--train-mb", "80", "--val-mb", "4"])
        if SMOKE_ONLY:
            build_sft_cmd.append("--smoke-quota")
        b_proc = subprocess.Popen(
            build_sft_cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert b_proc.stdout is not None
        for line in b_proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
        if int(b_proc.wait()) != 0:
            raise RuntimeError("Stage 5 SFT dataset build failed!")
        tmp_tar = NVME_CACHE / "sft_stage5.tar.gz"
        with tarfile.open(tmp_tar, "w:gz") as tf:
            for fpath in sorted(NVME_SFT_DIR.iterdir()):
                if fpath.is_file():
                    tf.add(fpath, arcname=fpath.name)
        sync_file_to_drive(tmp_tar, DRIVE_SFT_ARCHIVE.parent, dest_name=DRIVE_SFT_ARCHIVE.name)
        tmp_tar.unlink(missing_ok=True)

    # 2. Run Stage 5 SFT (15,000 steps for v2 cognitive 210M / 8,000 steps for v1 48M)
    sft_cmd = [
        sys.executable,
        "-u",
        "train/train_stage5_sft.py",
        "--gpus", "1",
        "--weights", str(BASE_WEIGHTS),
        "--data-dir", str(NVME_SFT_DIR),
        "--cap-data-dir", str(NVME_CAP_DIR),
        "--export-dir", str(NVME_CACHE / "export"),
        "--drive-export-dir", str(DRIVE_EXPORT),
        "--total-steps", SFT_STEPS,
        "--target-tokens-bytes-per-step", "65536",
        "--max-doc-bytes", MAX_DOC_BYTES,
        "--compile-mode", "auto",
    ]
    if BASE_CONFIG.exists():
        sft_cmd.extend(["--config", str(BASE_CONFIG)])
    print(f"[Stage 5 SFT] Launching: {' '.join(sft_cmd)}", flush=True)
    proc = subprocess.Popen(sft_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    exit_code = int(proc.wait())
    if exit_code == 0 and CHAT_WEIGHTS.exists():
        print(f"[Stage 5 DONE] Chat + Tool SFT weights synced to Drive: {CHAT_WEIGHTS}", flush=True)
    else:
        raise RuntimeError(f"Stage 5 SFT exited with code {exit_code}")
finally:
    if hasattr(os, "sync"):
        os.sync()
    if AUTO_UNASSIGN_RUNTIME_ON_FINISH and exit_code == 0 and CHAT_WEIGHTS.exists():
        print("[Auto-Unassign] Waiting 10s after os.sync() before releasing GPU...", flush=True)
        time.sleep(10)
        from google.colab import runtime

        runtime.unassign()
